# Full Benchmark Pipeline: Single File Example

This notebook walks through the complete benchmark pipeline described in
[docs/replicating-paper-results.md](../docs/replicating-paper-results.md)
using a **single example file** (`FAV70669_117da01a_45f6321d_0.pod5`) from
dataset DS1, so every step can be reproduced on any supported machine without
downloading the full corpus.

The pipeline covers the following benchmark types:

| Step | Benchmark | Input format | Paper reference |
|------|-----------|--------------|----------------|
| 1 | Download example file | N/A | replicating-paper-results §1 |
| 2 | Generate `.bin` files | POD5 | replicating-paper-results §2–3 |
| 3 | Compression benchmark | `.bin` | replicating-paper-results §4 |
| 4 | Speed benchmark | `.bin` | replicating-paper-results §5 |
| 5 | Size-split benchmark | POD5 | replicating-paper-results §7 |
| 6 | Time-split benchmark | POD5 | replicating-paper-results §7 |

> **Note on the memory benchmark:** The memory benchmark (step 6 in the full
> pipeline) is intentionally excluded here. It is Linux-only and requires the
> separate `copy` executable. Refer to
> [docs/benchmark_binaries_compilation.md](../docs/benchmark_binaries_compilation.md)
> for the optional `copy` build target and to
> [docs/running-benchmarks.md](../docs/running-benchmarks.md) for
> the memory benchmark workflow.

---

**Before running this notebook:**
- Follow [README.md](../README.md) to create and activate the `ncb-build` conda
  environment.
- Build the benchmark binaries following
  [docs/benchmark_binaries_compilation.md](../docs/benchmark_binaries_compilation.md).
- Run all cells in order; each step depends on the outputs of the previous one.

In [ ]:
from __future__ import annotations

import importlib
import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    candidate = (start or Path.cwd()).resolve()
    for path in [candidate, *candidate.parents]:
        if (path / "CMakeLists.txt").is_file() and (path / "scripts").is_dir():
            return path
    raise FileNotFoundError(
        "Could not locate the repository root from the current working directory."
    )


REPO_ROOT = find_repo_root()

# Configure matplotlib cache directory to stay within the repo.
matplotlib_cache_dir = REPO_ROOT / ".matplotlib"
matplotlib_cache_dir.mkdir(exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(matplotlib_cache_dir))

SCRIPTS_BENCHMARKS = REPO_ROOT / "scripts" / "benchmarks"
SCRIPTS_UTILS = REPO_ROOT / "scripts" / "utils"

for _import_path in (SCRIPTS_BENCHMARKS, SCRIPTS_UTILS):
    if str(_import_path) not in sys.path:
        sys.path.insert(0, str(_import_path))

# Validate that all required Python packages are installed in this kernel.
required_modules = {
    "matplotlib": "matplotlib",
    "numpy": "numpy",
    "pandas": "pandas",
    "pod5": "pod5",
    "awscli": "awscli",
    "seaborn": "seaborn",
}
missing_modules = [
    pkg
    for pkg, mod in required_modules.items()
    if importlib.util.find_spec(mod) is None
]
if missing_modules:
    raise ModuleNotFoundError(
        "Install the missing notebook dependencies in the active kernel: "
        + ", ".join(sorted(missing_modules))
    )

display = importlib.import_module("IPython.display").display
plt = importlib.import_module("matplotlib.pyplot")
pd = importlib.import_module("pandas")
sns = importlib.import_module("seaborn")

# Benchmark runners and utilities
benchmark_algorithms = importlib.import_module("common.algorithms")
benchmark_executables = importlib.import_module("common.executables")
preprocess = importlib.import_module("process_files_for_time_benchmark")
compression_benchmark_runner = importlib.import_module("run_compression_benchmark")
speed_benchmark_runner = importlib.import_module("run_speed_benchmark")
run_size_split_benchmark = importlib.import_module("run_size_split_benchmark")
run_time_split_benchmark = importlib.import_module("run_time_split_benchmark")
analysis_speed = importlib.import_module("analysis.speed")
analysis_splits = importlib.import_module("analysis.splits")

build_speed_artifacts_from_frames = analysis_speed.build_speed_artifacts_from_frames
plot_tradeoff_bits_per_sample_vs_speed = (
    analysis_speed.plot_tradeoff_bits_per_sample_vs_speed
)
build_split_artifacts_from_frames = analysis_splits.build_split_artifacts_from_frames
plot_split_decomposition = analysis_splits.plot_split_decomposition

CANONICAL_ALGORITHM_ORDER = benchmark_algorithms.CANONICAL_ALGORITHM_ORDER
pd.set_option("display.max_columns", 50)

plt.style.use("default")
plt.rcParams["figure.figsize"] = (12, 7)
plt.rcParams["font.size"] = 10
sns.set_theme(style="whitegrid")

print(f"Repository root: {REPO_ROOT}")

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
EXAMPLE_POD5_DIR = REPO_ROOT / "data" / "pod5" / "ExamplePod5"
EXAMPLE_BIN_DIR = REPO_ROOT / "data" / "benchmark_bin" / "ExampleBin"
CONVERTER_SCRIPT = REPO_ROOT / "scripts" / "utils" / "pod5_to_benchmark_time.py"
NOTEBOOK_OUTPUT_DIR = REPO_ROOT / "notebooks" / "outputs" / "example_full_pipeline"
NOTEBOOK_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── .bin generation settings ───────────────────────────────────────────────────
# One process and one sample per chunk keeps the example fast.
BIN_GENERATION_PROCESSES = 1
BIN_GENERATION_SAMPLE_SIZE = 1
# Set to True to delete any previously generated .bin files before regenerating.
RESET_EXAMPLE_BIN = True

# ── Algorithms ─────────────────────────────────────────────────────────────────
# These are the three algorithms used in the paper for most benchmarks.
# To run all supported algorithms, replace with: list(CANONICAL_ALGORITHM_ORDER)
ALGORITHMS = ["VBZ", "EX-ZD-ZSTD", "PDZ"]
EXAMPLE_MACHINE_LABEL = "Example machine"
FIGURE_EXPORT_KWARGS = {
    "export_enabled": False,
    "output_dir": NOTEBOOK_OUTPUT_DIR,
    "export_format": "png",
    "include_titles": True,
}

# ── Pre-flight checks ─────────────────────────────────────────────────────────
EXAMPLE_BIN_DIR.mkdir(parents=True, exist_ok=True)
if not CONVERTER_SCRIPT.is_file():
    raise FileNotFoundError(f"Converter script not found: {CONVERTER_SCRIPT}")

compression_executable = benchmark_executables.resolve_named_executable(
    "compression_benchmark"
)
speed_executable = benchmark_executables.resolve_named_executable("speed_benchmark")
time_split_executable = benchmark_executables.resolve_named_executable(
    "time_split_benchmark"
)

config_rows = [
    {"setting": "EXAMPLE_POD5_DIR", "value": str(EXAMPLE_POD5_DIR)},
    {"setting": "EXAMPLE_BIN_DIR", "value": str(EXAMPLE_BIN_DIR)},
    {"setting": "CONVERTER_SCRIPT", "value": str(CONVERTER_SCRIPT)},
    {"setting": "NOTEBOOK_OUTPUT_DIR", "value": str(NOTEBOOK_OUTPUT_DIR)},
    {"setting": "ALGORITHMS", "value": ", ".join(ALGORITHMS)},
    {"setting": "EXAMPLE_MACHINE_LABEL", "value": EXAMPLE_MACHINE_LABEL},
    {
        "setting": "FIGURE_EXPORT_ENABLED",
        "value": str(FIGURE_EXPORT_KWARGS["export_enabled"]),
    },
    {"setting": "compression_benchmark", "value": str(compression_executable)},
    {"setting": "speed_benchmark", "value": str(speed_executable)},
    {"setting": "time_split_benchmark", "value": str(time_split_executable)},
]
display(pd.DataFrame(config_rows))

## Step 1: Download the Example Dataset

The full paper pipeline begins by downloading all ten datasets (DS1–DS10).
Here we download a single representative file, `FAV70669_117da01a_45f6321d_0.pod5`, from dataset **DS1**
(Oxford Nanopore open data, *D. melanogaster* R10.4.1
400 bps run).

The file is fetched via the AWS S3 CLI from the public prefix:
`s3://ont-open-data/contrib/melanogaster_bkim_2023.01/flowcells/D.melanogaster.R1041.400bps/`

It is placed in `data/pod5/ExamplePod5/`, which is the directory that all
subsequent benchmark steps read from.

See [docs/replicating-paper-results.md (Step 1)](../docs/replicating-paper-results.md)
for the equivalent command that downloads the entire corpus, and the `--dataset`
options accepted by the download script.

> **Prerequisite:** The AWS CLI must be installed and reachable on `$PATH`.
> No credentials are required; the bucket is public (`--no-sign-request`).

In [ ]:
import shutil

pod5_files = (
    sorted(EXAMPLE_POD5_DIR.glob("*.pod5")) if EXAMPLE_POD5_DIR.is_dir() else []
)

if not pod5_files:
    # Check for the AWS CLI before attempting the download.
    if shutil.which("aws") is None:
        raise EnvironmentError(
            "The AWS CLI ('aws') was not found on PATH.\n"
            "Datasets DS1\u2013DS5 and the example file are hosted on public AWS S3 "
            "and require the AWS CLI to download (no credentials needed).\n\n"
            "Install it with one of the following:\n"
            "  macOS (Homebrew):  brew install awscli\n"
            "  conda environment: conda install -c conda-forge awscli\n"
            "  Official installer: https://docs.aws.amazon.com/cli/latest/userguide/getting-started-install.html\n\n"
            "After installing, restart the kernel and re-run this cell."
        )

    print(f"No POD5 files found in {EXAMPLE_POD5_DIR}. Downloading...")
    download_script = REPO_ROOT / "data" / "pod5" / "download_dataset.py"
    subprocess.run(
        [sys.executable, str(download_script), "--dataset", "example"],
        check=True,
    )
    pod5_files = sorted(EXAMPLE_POD5_DIR.glob("*.pod5"))
else:
    print(
        f"Found {len(pod5_files)} existing POD5 file(s) in {EXAMPLE_POD5_DIR}. Skipping download."
    )

if not pod5_files:
    raise RuntimeError(
        f"No POD5 files found in {EXAMPLE_POD5_DIR} after download attempt. "
        "Check that the AWS CLI is installed and the download completed successfully."
    )

display(
    pd.DataFrame(
        [{"filename": f.name, "size_bytes": f.stat().st_size} for f in pod5_files]
    )
)

## Step 2: Generate `.bin` Files

The compression and speed benchmarks operate on raw signal data stored in a
flat binary format (`.bin`). Each `.bin` file contains one chunk of signal
samples extracted from a POD5 read, ready to be passed directly to the C++
benchmark executables.

The conversion is performed by `scripts/utils/process_files_for_time_benchmark.py`,
which reads POD5 reads, extracts signal samples, and writes one `.bin` file per
chunk.

In the full paper pipeline, this step corresponds to:
- [Step 2](../docs/replicating-paper-results.md): generating full `.bin` files
  for the compression benchmark.
- [Step 3](../docs/replicating-paper-results.md): selecting the speed-benchmark
  subset (first file per dataset, ≤ ~200 MB).

For this example we use a single file, so the same `.bin` output is used for
both the compression and speed benchmarks.

See [docs/running-benchmarks.md](../docs/running-benchmarks.md) for a description
of the `.bin` format and how it is consumed by each benchmark.

In [ ]:
if RESET_EXAMPLE_BIN:
    stale_bins = list(EXAMPLE_BIN_DIR.rglob("*.bin"))
    for stale in stale_bins:
        stale.unlink()
    if stale_bins:
        print(f"Removed {len(stale_bins)} stale .bin file(s) from {EXAMPLE_BIN_DIR}")

generated_pairs = preprocess.process_directory(
    str(EXAMPLE_POD5_DIR),
    str(EXAMPLE_BIN_DIR),
    BIN_GENERATION_PROCESSES,
    str(CONVERTER_SCRIPT),
    BIN_GENERATION_SAMPLE_SIZE,
)
generated_bin_paths = sorted(Path(out) for _, out in generated_pairs)

if not generated_bin_paths:
    raise RuntimeError(
        f"No .bin files were generated from {EXAMPLE_POD5_DIR}. "
        "Ensure the POD5 file was downloaded and the converter script is functional."
    )

print(f"Generated {len(generated_bin_paths)} .bin file(s) in {EXAMPLE_BIN_DIR}")
display(
    pd.DataFrame(
        {
            "bin_file": [p.name for p in generated_bin_paths],
            "relative_path": [
                str(p.relative_to(EXAMPLE_BIN_DIR)) for p in generated_bin_paths
            ],
            "size_bytes": [p.stat().st_size for p in generated_bin_paths],
        }
    )
)

## Step 3: Compression Benchmark

The compression benchmark measures two things for each algorithm:
- **Bits per sample:** how tightly the signal is compressed.
- **Correctness:** whether each decompressed chunk is a bit-for-bit match of
  the original.

It operates on the `.bin` files generated in Step 2 and is run once across all
datasets; machine identity does not affect the result.

This corresponds to [Step 4](../docs/replicating-paper-results.md) of the full
pipeline. Detailed CLI usage and output structure are described in the
[compression benchmark section of docs/running-benchmarks.md](../docs/running-benchmarks.md).

The runner writes output under
`results/generated/<timestamp>_<algorithms>_<input>/`, with per-file raw CSVs
in `raw/` and per-dataset summary CSVs in `summaries/`. The key summary columns
are `bits_per_sample` and `success_rate`.

In [ ]:
compression_result = compression_benchmark_runner.run_pipeline(
    str(EXAMPLE_BIN_DIR),
    ALGORITHMS,
)

print(f"Run ID:     {compression_result['run_id']}")
print(f"Run root:   {compression_result['run_root']}")
print(f"Summaries:  {compression_result['summaries_root']}")

compression_frames = []
for algorithm, summary_files in compression_result["summary_outputs"].items():
    for sf in summary_files:
        frame = pd.read_csv(sf)
        if frame.empty:
            continue
        frame["algorithm"] = algorithm
        compression_frames.append(frame)

if not compression_frames:
    raise RuntimeError("No compression summary CSVs were produced.")

compression_df = pd.concat(compression_frames, ignore_index=True)
display(
    compression_df[
        ["algorithm", "File", "bits_per_sample", "success_rate", "total_chunks"]
    ]
)

## Step 4: Speed Benchmark

The speed benchmark measures compression and decompression **throughput in MB/s**
for each algorithm. Unlike the compression benchmark, results are
machine-dependent. The paper reports results from multiple hardware
configurations.

It takes the same `.bin` files as input. The runner serialises all algorithm
runs intentionally (no `-j` flag) to avoid measurement interference between
concurrent processes.

This corresponds to [Step 5](../docs/replicating-paper-results.md) of the full
pipeline. Full CLI usage is described in the
[speed benchmark section of docs/running-benchmarks.md](../docs/running-benchmarks.md).

Output is written to `results/generated/speed/<run-id>/`. The summary CSV
columns `compression_speed_mb_sec`, `decompression_speed_mb_sec`, and
`bits_per_sample` are combined below into the same tradeoff-style figure used
in the article, but for a single machine.

In [ ]:
speed_result = speed_benchmark_runner.run_pipeline(
    str(EXAMPLE_BIN_DIR),
    ALGORITHMS,
)

speed_run_dir = Path(speed_result["summaries_root"]).parent

print(f"Run ID:     {speed_result['run_id']}")
print(f"Run root:   {speed_run_dir}")
print(f"Summaries:  {speed_result['summaries_root']}")

speed_frames = []
for algorithm, summary_files in speed_result["summary_outputs"].items():
    for sf in summary_files:
        frame = pd.read_csv(sf)
        if frame.empty:
            continue
        frame["algorithm"] = algorithm
        speed_frames.append(frame)

if not speed_frames:
    raise RuntimeError("No speed summary CSVs were produced.")

speed_df = pd.concat(speed_frames, ignore_index=True)
speed_context = build_speed_artifacts_from_frames(
    speed_df,
    machine=EXAMPLE_MACHINE_LABEL,
    requested_algorithms=ALGORITHMS,
    run_dir=speed_run_dir,
    run_id=speed_result["run_id"],
    dataset_name=EXAMPLE_BIN_DIR.name,
)
speed_tradeoff_df = plot_tradeoff_bits_per_sample_vs_speed(
    speed_context["comparison_df"],
    speed_context["algorithms"],
    speed_context["machine_order"],
    **FIGURE_EXPORT_KWARGS,
)

display(
    speed_tradeoff_df[
        [
            "machine",
            "algorithm",
            "avg_bits_per_sample",
            "avg_compression_speed",
            "avg_decompression_speed",
        ]
    ].round(4)
)

## Step 5: Size-Split Benchmark

The size-split benchmark analyses POD5 files directly (not `.bin`) to answer:
*what fraction of each file is occupied by compressed signal data vs. signal
table overhead vs. everything else?*

For each POD5 file it reports:
- `compressed_signal_bytes`: bytes used by compressed signal data.
- `total_signal_table_bytes`: compressed signal + per-row table overhead
  (row indices + signal row offsets).
- `samples_pct`: fraction of the file taken up by compressed signal.
- `signal_table_pct`: fraction taken up by the full signal table.

This benchmark requires only the `pod5` Python package (no C++ executable).

In the full paper pipeline this is the first part of
[Step 7](../docs/replicating-paper-results.md), run on datasets DS2 and DS7.
Detailed usage is in the
[size-split section of docs/running-benchmarks.md](../docs/running-benchmarks.md).

Output is written to
`results/generated/size_split/<run-id>/summaries/pod5_size_summaries.csv`.

In [ ]:
size_split_csv = run_size_split_benchmark.pipeline(str(EXAMPLE_POD5_DIR))
print(f"Summary written to: {size_split_csv}")

size_split_df = pd.read_csv(size_split_csv)
display(
    size_split_df[
        [
            "filename",
            "total_bytes",
            "compressed_signal_bytes",
            "total_signal_table_bytes",
            "uncompressed_signal_bytes",
            "samples_pct",
            "signal_table_pct",
            "rows",
        ]
    ]
)

## Step 6: Time-Split Benchmark

The time-split benchmark measures what fraction of total compression and
decompression time is spent on **signal processing** (the codec itself) versus
everything else (I/O, Arrow serialisation, read metadata, etc.).

It uses the `time_split_benchmark` C++ executable and takes POD5 files as
input. For each file it runs two passes: one compressing and one decompressing,
and parses the C++ timing output.

Key output columns:
- `comp_proc_s` / `decomp_proc_s`: seconds spent in the codec.
- `comp_total_s` / `decomp_total_s`: total elapsed seconds.
- `comp_pct` / `decomp_pct`: codec fraction of total time (%).

In the full paper pipeline this is the second part of
[Step 7](../docs/replicating-paper-results.md), run on datasets DS2 and DS7.
Detailed usage is in the
[time-split section of docs/running-benchmarks.md](../docs/running-benchmarks.md).

Output is written to
`results/generated/time_split/<run-id>/summaries/pod5_benchmarks.csv`. Once both
split steps have run, the notebook combines them below into the same workload
decomposition figure used in the article.

In [ ]:
time_split_csv = run_time_split_benchmark.pipeline(str(EXAMPLE_POD5_DIR))
print(f"Summary written to: {time_split_csv}")

time_split_df = pd.read_csv(time_split_csv)
display(
    time_split_df[
        [
            "filename",
            "comp_total_s",
            "comp_proc_s",
            "comp_pct",
            "comp_intervals",
            "decomp_total_s",
            "decomp_proc_s",
            "decomp_pct",
            "decomp_intervals",
        ]
    ]
)

split_context = build_split_artifacts_from_frames(
    size_split_df,
    time_split_df,
    size_run_dir=Path(size_split_csv).parent.parent,
    time_run_dir=Path(time_split_csv).parent.parent,
)
split_overall_df = split_context["overall_df"].copy()

print("Size-split source run:", split_context["size_run_dir"])
print("Time-split source run:", split_context["time_run_dir"])

plot_split_decomposition(split_overall_df, **FIGURE_EXPORT_KWARGS)

display(
    split_overall_df[
        [
            "file_count",
            "compressed_signal_pct",
            "signal_table_overhead_pct",
            "remaining_file_pct",
            "comp_pct",
            "decomp_pct",
        ]
    ].round(4)
)

## Equivalent CLI Commands

All six steps can also be run directly from the command line. The commands below
assume you are in the repository root.

---

### Step 1: Download the example file

Downloads `FAV70669_117da01a_45f6321d_0.pod5` from DS1 into
`data/pod5/ExamplePod5/`.

```sh
python data/pod5/download_dataset.py --dataset example
```

Add `--dry-run` to preview without writing files.

---

### Step 2: Generate `.bin` files

```sh
python scripts/utils/process_files_for_time_benchmark.py \
    data/pod5/ExamplePod5 \
    data/benchmark_bin/ExampleBin \
    -p 1 --sample-size 1
```

---

### Step 3: Compression benchmark

```sh
python scripts/benchmarks/run_compression_benchmark.py \
    data/benchmark_bin/ExampleBin \
    VBZ EX-ZD-ZSTD PDZ
```

---

### Step 4: Speed benchmark

```sh
python scripts/benchmarks/run_speed_benchmark.py \
    data/benchmark_bin/ExampleBin \
    VBZ EX-ZD-ZSTD PDZ
```

---

### Step 5: Size-split benchmark

```sh
python scripts/benchmarks/run_size_split_benchmark.py \
    data/pod5/ExamplePod5
```

---

### Step 6: Time-split benchmark

```sh
python scripts/benchmarks/run_time_split_benchmark.py \
    data/pod5/ExamplePod5
```